# 90 · Del audio al análisis

## Propósito

Tomar una transcripción y pasarla por spaCy, que es lo que la unidad prometió
y no hizo. Vamos a encontrar errores de análisis lingüístico y a reconstruir
su procedencia: algunos nacen en la transcripción; otros aparecen cuando spaCy
interpreta un texto oral segmentado y puntuado por otro modelo.

La pregunta no es solamente si una herramienta “se equivoca”, sino qué etapa
produjo el problema, cómo se transforma al pasar a la etapa siguiente y en qué
capa conviene intervenir.

## Qué consume

- Un audio propio. Nosotros usamos una charla de historia argentina.
- `es_core_news_lg`, el mismo modelo de la unidad 4.

## Resultados de aprendizaje

- Explicar por qué un error de transcripción se propaga al análisis lingüístico.
- Auditar la salida del NER sobre habla transcripta, no sobre prosa escrita.
- Comparar tres formas de corregir el mismo error, en tres capas distintas.
- Decidir en qué capa conviene corregir, y poder defenderlo.
- Distinguir entre corregir el texto, corregir el reconocimiento de entidades
  y normalizar formas para un análisis posterior.
- Armar una red de coocurrencia de entidades sobre tu propio material.

> **Este cuaderno es complementario y corre en Google Colab.** No entra en la
> evaluación. Se hace después del `02`, con un audio que ya sepas transcribir.

## Dos recorridos posibles

Este cuaderno tiene un núcleo y una extensión. No hace falta llegar a la red
para haber entendido el problema principal.

### Recorrido mínimo

1. Transcribir el audio.
2. Pasar la transcripción por spaCy y auditar entidades.
3. Comparar tres capas de intervención:
   `initial_prompt`, diccionario de reemplazos y `EntityRuler`.
4. Documentar qué corrige, qué conserva y qué riesgo introduce cada capa.

### Recorrido extendido

Además del recorrido mínimo:

5. Normalizar identidades para análisis.
6. Visualizar una relación sintáctica.
7. Construir y criticar una red de coocurrencias por segmento.

La ruta extendida no es “mejor”: agrega nuevas decisiones, nuevas fuentes de
error y nuevas preguntas metodológicas.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **NER** | *Named Entity Recognition*: encontrar en un texto los nombres de personas, lugares y organizaciones |
| **entidad** | Cada uno de esos nombres, con una etiqueta: `PER`, `LOC`, `ORG`, `MISC` |
| **displaCy** | El visualizador de spaCy. Pinta entidades sobre el texto y dibuja árboles de sintaxis |
| **`initial_prompt`** | Un texto que se le pasa a Whisper *antes* de transcribir, para inclinarlo hacia cierto vocabulario |
| **`EntityRuler`** | Un componente de spaCy que reconoce entidades por reglas escritas a mano, no por estadística |
| **coocurrencia** | Dos entidades que aparecen en el mismo fragmento. Si pasa muchas veces, algo las relaciona |
| **límite de entidad** | Dónde empieza y termina el fragmento que spaCy toma como una entidad. Por ejemplo, `Arturo Jauretzi` tiene un límite distinto de `Dice Arturo Jauretzi` |
| **forma canónica** | Forma unificada con la que *decidimos* contar varias menciones como la misma entidad. Por ejemplo, contar `Sarmiento` y `Domingo Faustino Sarmiento` bajo una sola forma. No la descubre el modelo: la declaramos nosotros |
| **pipeline** | Cadena ordenada de componentes de spaCy. Cada componente recibe el texto y las anotaciones que dejaron los anteriores |

## De dónde viene este cuaderno

La unidad 5 terminó diciendo esto:

> *El texto transcripto no se parece al escrito. Tiene otra puntuación —la inventó el
> modelo—, repeticiones, muletillas y cortes. `es_core_news_lg` se entrenó con prensa.
> Pasale una transcripción y mirá qué hace.*

Eso es lo que vamos a hacer, y no sale bien. Ese es el punto.

**La analogía.** Imaginate que alguien te dicta por teléfono una lista de apellidos y vos
la anotás. Después le pasás tu lista a otra persona para que busque a esa gente en una
guía. Si vos escuchaste mal un apellido, la segunda persona no va a encontrar a nadie —y
no va a ser culpa suya, ni va a tener forma de darse cuenta.

Whisper es el que anota. spaCy es el que busca en la guía.

La analogía tiene un límite: si Whisper anotó mal un apellido, spaCy no puede
recuperar automáticamente lo que se dijo en el audio. Pero incluso si el nombre
llegara bien escrito, spaCy puede equivocarse al decidir dónde empieza, dónde
termina o qué tipo de entidad es, especialmente sobre habla transcripta.

**Dónde vive esto.** Cualquier sistema que encadene modelos tiene este problema: la
transcripción de una reunión que alimenta un resumen, el OCR de un documento que alimenta
una búsqueda, el subtítulo automático que alimenta un buscador de video. **El error de la
primera etapa no se anuncia: se hereda.**
Y al heredarse puede cambiar de forma: una grafía deformada puede convertirse
en una entidad ausente, dos personas pueden convertirse en una sola entidad,
o una puntuación agregada por el ASR puede cambiar cómo spaCy delimita una oración.

## ⚙ Entorno

Dos cosas: la librería para transcribir y el modelo de español de spaCy. Colab ya trae
spaCy instalado, así que solo falta bajarle el modelo.

> **Reproducibilidad.** Colab y sus bibliotecas cambian con el tiempo. Al inicio
> del cuaderno vamos a registrar las versiones y el pipeline disponible para poder
> saber con qué entorno se obtuvo una salida.

In [ ]:
%pip -q install faster-whisper
!python -m spacy download es_core_news_lg

print("Instalación terminada. Reiniciá el entorno si Colab te lo pide.")

Y ahora las importaciones. Son las mismas de la unidad 4 —spaCy, `Counter`, pandas,
matplotlib— más `displacy`, que es el visualizador.

`numpy` aparece solo al final, para repartir los nodos de la red sobre un círculo.

`importlib.metadata` sirve para preguntarle a Python qué versión quedó instalada de un
paquete. La usamos para `faster-whisper`, que no expone su versión de forma tan directa
como spaCy.

In [ ]:
import spacy
from spacy import displacy
from collections import Counter
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

nlp = spacy.load("es_core_news_lg")

# Registramos el entorno: con otras versiones, las salidas pueden cambiar.
version_faster_whisper = importlib.metadata.version("faster-whisper")
print("Versión de spaCy:", spacy.__version__)
print("Versión de faster-whisper:", version_faster_whisper)
print("Pipeline cargado:", nlp.pipe_names)

**Fijate en `nlp.pipe_names`.** Ahí está `ner`, que es el componente que vamos a
auditar. Es el mismo modelo que usaste toda la unidad 4, sin ningún cambio.

Si compartís una salida de este cuaderno, compartí también esas tres líneas: son la
forma de saber con qué entorno se produjo.

## El audio

Subí un audio hablado en español. Una entrevista, una charla, una clase: algo con nombres
propios adentro, que es lo que vamos a mirar.

Nosotros usamos una charla de historia argentina sobre Sarmiento, que tiene decenas de
nombres de personas y de lugares. Cuanto más densa en nombres, mejor sale el ejercicio.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()

nombres_subidos = list(archivos_subidos.keys())
AUDIO = nombres_subidos[0]

print("Vamos a transcribir:", AUDIO)

> **Dos advertencias, las mismas del `00` y del `01`.**
>
> **El disco de Colab es prestado.** Lo que subas y lo que generes desaparece cuando se
> cierra la sesión. Si querés conservar algo, descargalo antes de irte.
>
> **Usá material que puedas usar.** Audio propio, público o con permiso explícito. Y no
> transcribas grabaciones de clase con intervenciones de tus compañeros: convertir esa voz
> en texto la vuelve buscable y copiable, y nadie dio permiso para eso.

Ahora la transcripción, igual que en el `01`. Si hay GPU va a tardar un par de
minutos; en CPU, bastante más.

In [ ]:
from faster_whisper import WhisperModel
import torch

HAY_GPU = torch.cuda.is_available()
if HAY_GPU:
    modelo_audio = WhisperModel(model_size_or_path="large-v3-turbo",
                                device="cuda", compute_type="float16")
else:
    modelo_audio = WhisperModel(model_size_or_path="small",
                                device="cpu", compute_type="int8")

print("GPU disponible:", HAY_GPU)

Con el modelo cargado, pedimos la transcripción. Los parámetros son los mismos que
justificaste en la tabla del `01`: español, filtro de voz, sin arrastrar contexto entre
segmentos.

Acordate de que esta llamada vuelve enseguida y todavía no transcribió nada.

In [ ]:
generador, info = modelo_audio.transcribe(
    audio=AUDIO,
    language="es",
    vad_filter=True,
    condition_on_previous_text=False,
    beam_size=5,
)

print("Llamada hecha. Falta recorrer el generador.")

Acá sí trabaja. Esta celda es la que tarda.

In [ ]:
segmentos = []

for segmento in generador:
    texto_limpio = segmento.text.strip()
    segmentos.append({"inicio": segmento.start, "fin": segmento.end,
                      "texto": texto_limpio})

cantidad_de_segmentos = len(segmentos)
print("Segmentos:", cantidad_de_segmentos)

Y el texto corrido, que es lo que spaCy necesita. Un texto, no una lista de
segmentos.

In [ ]:
textos_de_los_segmentos = []

for segmento in segmentos:
    textos_de_los_segmentos.append(segmento["texto"])

texto_completo = " ".join(textos_de_los_segmentos)

cantidad_de_caracteres = len(texto_completo)
print("Caracteres transcriptos:", cantidad_de_caracteres)

## Primera pasada

Le damos el texto a spaCy y miramos qué entidades encuentra. Es la misma operación del
cuaderno `03` de la unidad 4, sobre otro material.

In [ ]:
documento = nlp(texto_completo)

entidades_encontradas = len(documento.ents)
print("Entidades detectadas:", entidades_encontradas)

`documento` es un objeto `Doc`, el mismo de la unidad 4: el texto más todo lo que
spaCy le calculó encima. `documento.ents` son las entidades que encontró.

**Las etiquetas son cuatro**, y conviene tenerlas a mano porque las vamos a filtrar:

| Etiqueta | Qué marca |
|---|---|
| `PER` | Personas |
| `LOC` | Lugares: ciudades, provincias, países |
| `ORG` | Organizaciones: empresas, instituciones, medios |
| `MISC` | El cajón de sastre: obras, eventos, nacionalidades |

Un periódico como *El Zonda* debería caer en `ORG`. Un pueblo como Chivilcoy, en `LOC`.
Anotá esa expectativa ahora, porque en un rato vamos a ver qué pasó de verdad.

Y ahora displaCy. `style="ent"` pinta las entidades sobre el texto, con su etiqueta
al lado.

Tomamos solo los primeros 2.000 caracteres: el texto entero no entra en pantalla y el
gráfico se vuelve inútil.

In [ ]:
primer_tramo = nlp(texto_completo[:2000])

displacy.render(primer_tramo, style="ent", jupyter=True)

**Se ve impecable.** Nombres resaltados, etiquetas de colores, todo prolijo.

Ese es exactamente el problema: una salida con buena cara no dice nada sobre si está bien.
Lo mismo que viste en toda la unidad 4.

## La auditoría

Contemos. Cuántas veces aparece cada persona, ordenadas de mayor a menor.

In [ ]:
menciones_de_personas = []

for entidad in documento.ents:
    if entidad.label_ == "PER":
        menciones_de_personas.append(entidad.text)

conteo_de_personas = Counter(menciones_de_personas)

for nombre, veces in conteo_de_personas.most_common(20):
    print(veces, "·", nombre)

Filtramos por `PER` para que la lista sea legible. `Counter` es la misma herramienta
de la unidad 3: recibe una lista y devuelve cuántas veces aparece cada cosa.

`most_common(20)` los ordena de mayor a menor y se queda con los primeros veinte.

### ✎ Para pensar

Mirá esa lista con atención antes de seguir.

- ¿Hay dos entradas que son **la misma persona** escrita distinto?
- ¿Hay algún nombre que no exista, que no sea una persona real?
- ¿Falta alguien que vos sabés que el audio nombra?

## Los nombres que el audio rompió

En nuestra charla sobre Sarmiento pasó esto:

> **Resultado de referencia, no salida garantizada.** Esta tabla documenta errores
> observados en una ejecución de referencia sobre una charla específica de historia
> argentina. Con otro audio, otro modelo, otra versión, otro dispositivo o una
> configuración diferente pueden aparecer errores distintos.

| Lo que escribió Whisper | Lo que se dijo |
|---|---|
| "Alberti" y "Juan Bautista Alverde" | **Alberdi** — la misma persona, dos veces, mal las dos |
| "el Sonda" | **El Zonda**, el periódico que fundó Sarmiento |
| "Chivicol" | **Chivilcoy**, el pueblo que fundó |
| "Arturo Jauretzi" | **Jauretche** |
| "Almacio Vélez Sárfiel" | **Dalmacio Vélez Sarsfield** |
| "Unzuea", "Leluar" | **Unzué**, **Leloir** |

**Estos errores no nacen en el NER.** spaCy encontró algo que parece un nombre propio,
en la posición donde va un nombre propio, y lo etiquetó como persona. El nombre ya estaba
mal escrito antes de que spaCy lo viera, y spaCy no tiene acceso al audio para recuperar
lo que se dijo. El error es heredado.

Y fijate la consecuencia: **en la ejecución de referencia, Alberdi aparece dos veces en el
conteo, con dos grafías**. Si
contaras menciones para decir quién es más importante en el relato, Alberdi quedaría
partido al medio y bajaría de puesto. Es el mismo problema de *Mirta / Bertotti / Mirta
Bertotti* del cuaderno `04`, con una causa nueva.

## Y hay un segundo error, peor que el primero

Mirá otra vez tu lista. Además de los nombres mal escritos, en la ejecución de referencia
aparecieron cosas como estas:

| Lo que detectó el NER | Qué pasó |
|---|---|
| `Dice Arturo Jauretzi` | Se comió el verbo adentro del nombre |
| `Gobernaba Rosas` | Lo mismo |
| `Urquiza Sarmiento` | **Dos personas distintas pegadas en una sola entidad** |
| `Juan Bautista Alverde Decíamos` | Se llevó puesta la palabra siguiente |
| `Quieren` | Un verbo solo, etiquetado como persona |
| `Estados Unidos`, `los Estados Unidos`, `en Estados Unidos` | El mismo lugar, contado tres veces |

**Esto no es un error de ortografía: es un error de límites.** El NER no solo tiene que
decir *qué* es una entidad, también dónde empieza y dónde termina. Y para eso se apoya
muchísimo en las mayúsculas y en los puntos.

En un texto escrito, un punto seguido de mayúscula marca el arranque de una oración. En una
transcripción **la puntuación la inventó Whisper**: donde no puso el punto, la palabra
siguiente queda en mayúscula en medio de una oración, y el modelo la lee como parte del
nombre propio anterior.

Es exactamente el error de `EE.UU.` pegado a la oración siguiente que auditaste en el `04`.
Mismo error, causa nueva.

**Este error no es de una sola capa: es de interacción.** Whisper decidió dónde poner
puntos y mayúsculas; el NER decidió los límites apoyándose en esas marcas. Ninguna de las
dos etapas lo produce sola: aparece cuando un modelo entrenado con prosa escrita analiza
habla puntuada por otro modelo.

## Tres lugares donde corregir

Acá está el corazón del cuaderno. El mismo error se puede atacar en tres momentos
distintos, y **ninguno es el correcto**: son tres decisiones con consecuencias distintas.

| | Cuándo actúa | Qué hace | Qué arriesga |
|---|---|---|---|
| **1. `initial_prompt`** | Antes de que el texto exista | Le avisa a Whisper qué nombres va a escuchar | No garantiza nada, y puede **meter** nombres que nadie dijo |
| **2. Diccionario** | Después, sobre el texto | Reemplaza `"Alverde"` por `"Alberdi"` | Ciego al contexto: pisa casos legítimos |
| **3. `EntityRuler`** | Después, sobre el análisis | Reglas dentro del pipeline de spaCy | Arregla la entidad, no el texto: la transcripción sigue mal |

Vamos a probar las tres.

## Capa 1: avisarle a Whisper

El `01` cerró con un experimento opcional: `initial_prompt`, un texto que se le pasa al
modelo antes de transcribir. Ahí quedó una pregunta abierta —*"probá un glosario con
términos que no estén en el audio: ¿aparecen igual en la salida?"*— y este es el momento de
contestarla.

**No es un diccionario de reemplazos.** Es un sesgo: le inclina la mano al modelo hacia
cierto vocabulario. Puede funcionar, puede no hacer nada, y puede empeorar las cosas.

In [ ]:
GLOSARIO = (
    "Charla de historia argentina. Nombres propios que aparecen: "
    "Domingo Faustino Sarmiento, Juan Bautista Alberdi, Juan Manuel de Rosas, "
    "Justo José de Urquiza, Bartolomé Mitre, Facundo Quiroga, "
    "Chivilcoy, El Zonda, Arturo Jauretche, Dalmacio Vélez Sarsfield."
)

print("Glosario de", len(GLOSARIO), "caracteres.")

La misma llamada de antes con un parámetro más. Todo lo demás queda igual: es la
única forma de saber que lo que cambie se debe al glosario.

In [ ]:
generador_con_glosario, info_con_glosario = modelo_audio.transcribe(
    audio=AUDIO,
    language="es",
    vad_filter=True,
    condition_on_previous_text=False,
    beam_size=5,
    initial_prompt=GLOSARIO,
)

print("Segunda transcripción pedida.")

Y ahora recorremos la segunda salida. Guardamos texto y tiempos.

El audio es el mismo, por lo que esperamos que los cortes temporales sean
parecidos. Pero no debemos suponer que un cambio en la decodificación preserve
exactamente los mismos límites de segmento: una salida puede dividir o unir
tramos de otra manera.

In [ ]:
segmentos_con_glosario = []

for segmento in generador_con_glosario:
    texto_limpio = segmento.text.strip()

    segmentos_con_glosario.append(
        {
            "inicio": segmento.start,
            "fin": segmento.end,
            "texto": texto_limpio,
        }
    )

cantidad_con_glosario = len(segmentos_con_glosario)

print("Segmentos con glosario:", cantidad_con_glosario)

## Comparación rápida de ambas transcripciones

Vamos a comparar los segmentos por su posición en cada lista. Esta comparación
sirve para inspeccionar cambios visibles, pero tiene un límite: si Whisper cambió
los cortes temporales, el segmento número 12 de una versión puede no corresponder
exactamente al segmento número 12 de la otra.

Por eso mostramos también los tiempos. No estamos haciendo una alineación formal
por solapamiento temporal; esa sería una extensión técnica posterior.

Solo se imprimen los primeros ocho segmentos que cambiaron.

In [ ]:
cantidad_a_comparar = min(
    len(segmentos),
    len(segmentos_con_glosario)
)

cambios_mostrados = 0

for posicion in range(cantidad_a_comparar):
    if cambios_mostrados >= 8:
        break

    segmento_sin_glosario = segmentos[posicion]
    segmento_con_glosario = segmentos_con_glosario[posicion]

    texto_sin_glosario = segmento_sin_glosario["texto"]
    texto_con_glosario = segmento_con_glosario["texto"]

    if texto_sin_glosario != texto_con_glosario:
        print(
            f"SIN [{segmento_sin_glosario['inicio']:.1f}"
            f"–{segmento_sin_glosario['fin']:.1f}]:"
        )
        print(texto_sin_glosario)

        print(
            f"CON [{segmento_con_glosario['inicio']:.1f}"
            f"–{segmento_con_glosario['fin']:.1f}]:"
        )
        print(texto_con_glosario)

        print()
        cambios_mostrados = cambios_mostrados + 1

Si los tiempos de un par no coinciden, esos dos segmentos no hablan del mismo tramo
de audio y la diferencia de texto no se puede atribuir solo al glosario.

Comparar textos segmento por segmento está bien para ver **qué** cambió. Pero la pregunta
real es si mejoró el análisis, y eso se ve pasando el NER otra vez.

In [ ]:
textos_con_glosario = []

for segmento in segmentos_con_glosario:
    textos_con_glosario.append(segmento["texto"])

texto_glosario = " ".join(textos_con_glosario)
documento_glosario = nlp(texto_glosario)

print("Entidades antes :", len(documento.ents))
print("Entidades ahora :", len(documento_glosario.ents))

Y el mismo tramo dibujado, para comparar contra el de más arriba con los ojos.

In [ ]:
primer_tramo_glosario = nlp(texto_glosario[:2000])

displacy.render(primer_tramo_glosario, style="ent", jupyter=True)

**Por qué esto funciona así, y por qué es frágil.**

Whisper genera el texto palabra por palabra, y en cada paso elige la más probable *dado lo
que ya escribió*. El `initial_prompt` se le pone adelante como si fuera texto que el modelo
ya hubiera producido. No es una instrucción ni una tabla de reemplazos: es **contexto
falso**, puesto ahí para inclinar las probabilidades.

Por eso puede corregir *Alverde* por *Alberdi* —esa secuencia ahora es más probable— y por
eso mismo puede escribir un nombre que nadie dijo, si vos se lo pusiste adelante. La
herramienta no distingue entre "ayudalo a escuchar bien" y "decile qué queremos oír".

### ✎ Para pensar

- ¿Corrigió los nombres? ¿Corrigió algunos y otros no?
- Si no cambió nada, **eso también es un resultado**: quiere decir que el sesgo no alcanzó
  para torcer lo que el modelo ya había decidido.
- Ahora la prueba que el `01` dejó pendiente: agregá al glosario un nombre que **no** esté
  en el audio —poné *Manuel Belgrano*— y volvé a correr. ¿Aparece en la transcripción?

Si aparece, acabás de ver el riesgo completo de esta capa: **le pediste al modelo que
esperara algo, y te lo escribió.**

## Capa 2: el diccionario

Esto es lo que la intuición pide primero, y es lo que mucha gente cree que hace el
glosario: una tabla de reemplazos sobre el texto ya escrito.

In [ ]:
CORRECCIONES = {
    "Alverde": "Alberdi",
    "Alberti": "Alberdi",
    "Chivicol": "Chivilcoy",
    "Jauretzi": "Jauretche",
    "Sárfiel": "Sarsfield",
    "Almacio": "Dalmacio",
}

print("Reemplazos definidos:", len(CORRECCIONES))

`str.replace` cambia todas las apariciones de un texto por otro. Lo aplicamos una
vez por cada entrada del diccionario, y después volvemos a pasar spaCy sobre el resultado.

Fijate que **el texto original no se toca**: `texto_completo` sigue siendo la
transcripción original de Whisper, y el texto intervenido por diccionario se guarda en
otra variable, `texto_corregido`. Perder el original sería perder la evidencia.

In [ ]:
texto_corregido = texto_completo

for forma_rota, forma_correcta in CORRECCIONES.items():
    texto_corregido = texto_corregido.replace(forma_rota, forma_correcta)

documento_corregido = nlp(texto_corregido)

menciones_corregidas = []
for entidad in documento_corregido.ents:
    if entidad.label_ == "PER":
        menciones_corregidas.append(entidad.text)

conteo_corregido = Counter(menciones_corregidas)
print("Alberdi ahora aparece:", conteo_corregido["Alberdi"], "veces")

**Funciona.** Y es peligroso por dos razones que conviene ver antes de adoptarlo.

**Es ciego al contexto.** `replace` no sabe de palabras: cambia cualquier coincidencia, esté
donde esté. Si un apellido roto coincide con una sílaba de otra palabra, la rompe también.

**Miente sobre el material.** El texto corregido ya no es la transcripción: es tu versión de
la transcripción. Si después alguien compara tu texto con el audio, no van a coincidir, y no
va a haber ningún registro de qué tocaste.

La frase siguiente es un ejemplo sintético: no intenta reproducir una salida
del audio, sino mostrar por qué un reemplazo textual puede acertar en un caso
y ser inadecuado en otro.

In [ ]:
frase_de_prueba = "El Sonda salió en San Juan. La sonda espacial llegó a Marte."

frase_reemplazada = frase_de_prueba.replace("Sonda", "Zonda")

print("Antes :", frase_de_prueba)
print("Después:", frase_reemplazada)

Ahí `replace` acertó por casualidad: respetó la minúscula. Probá el mismo reemplazo
sin distinguir mayúsculas y vas a convertir la sonda espacial en un viento sanjuanino.

**Si usás esta capa, el reemplazo va documentado**: qué cambiaste y por qué. Es la misma
regla del `FUENTE.md` de la unidad 2.

## Capa 3: `EntityRuler`

La tercera forma no toca el texto: mete reglas adentro del pipeline de spaCy. Es lo que
hiciste en el `05` de la unidad 4.

`before="ner"` lo pone **antes** del reconocedor estadístico, y `overwrite_ents=True` hace
que la regla gane cuando choca con el modelo.

In [ ]:
nlp_mejorado = spacy.load("es_core_news_lg")

configuracion_de_reglas = {"overwrite_ents": True}
reglas = nlp_mejorado.add_pipe("entity_ruler",
                               config=configuracion_de_reglas,
                               before="ner")

print("Pipeline con entity_ruler:")
print(nlp_mejorado.pipe_names)

**Dos detalles de esa celda que deciden todo.**

`before="ner"` ubica nuestras reglas antes del reconocedor estadístico. De ese
modo, el pipeline ve primero las entidades que declaramos por patrón.

`overwrite_ents=True` indica que, cuando una regla reconoce un tramo de texto,
puede tener prioridad sobre una entidad existente que se superponga.

Podríamos diseñar otro orden de componentes. En este cuaderno elegimos este
orden para hacer visible que el pipeline es una cadena: el lugar que ocupa cada
componente también forma parte del diseño de la herramienta.

Ahora las reglas. Cada una dice: *este texto exacto es una entidad de esta
categoría*. Fijate que acá **no corregimos la grafía**: le enseñamos a spaCy que esa grafía
rota nombra a una persona o a un lugar.

**La lista va en un archivo, no en una celda.** Es lo mismo que hiciste con
`personajes.txt` en el `05`, y por la misma razón: quién entra y quién no **es una decisión
de análisis**, no una configuración. Tiene que poder leerse, editarse y citarse aparte del
código.

El formato es el de la unidad 4: una entrada por línea, `categoria|texto`, y las líneas que
empiezan con `#` se ignoran.

**Fijate en algo que parece un error y no lo es:** la lista incluye `Juan Bautista Alverde`,
`Chivicol` y `Arturo Jauretzi`, que están **mal escritos**. Están a propósito.

El `EntityRuler` no corrige ortografía: reconoce texto literal. Si queremos que entienda que
*"Alverde"* nombra a una persona, tenemos que ponerle exactamente esa cadena, tal como
aparece en la transcripción. Poner *Alberdi* bien escrito no serviría de nada, porque esa
secuencia no está en el texto.

**La lista se escribe contra el material que tenés, no contra el que te gustaría tener.**

In [ ]:
%%writefile nombres.txt
# Nombres propios de esta charla, para el EntityRuler.
#
# ESTA LISTA ES UNA DECISIÓN DE ANÁLISIS, NO UN ARCHIVO DE CONFIGURACIÓN.
# Incluye grafías que el ASR escribió mal: están acá justamente porque
# hay que reconocerlas. Editala y anotá tu criterio.
#
# Formato:  categoria|texto

PER|Sarmiento
PER|Domingo Faustino Sarmiento
PER|Rosas
PER|Juan Manuel de Rosas
PER|Urquiza
PER|Mitre
PER|Facundo Quiroga
PER|Juan Facundo Quiroga
PER|Chacho Peñalosa
PER|Alberti
PER|Juan Bautista Alberti
PER|Juan Bautista Alverde
PER|Arturo Jauretzi
PER|Almacio Vélez Sárfiel
PER|Aurelia Vélez
LOC|San Juan
LOC|Buenos Aires
LOC|Chivicol
LOC|Estados Unidos
LOC|Chile
LOC|Paraguay
ORG|el Sonda
ORG|El Censor

Y ahora la leemos, con el mismo bucle del `05`. Las líneas de comentario se
descartan y cada entrada queda como un par `(categoría, texto)`.

In [ ]:
ENTRADAS = []

with open("nombres.txt", encoding="utf-8") as archivo:
    for linea in archivo:
        linea = linea.strip()
        if "|" in linea:
            if not linea.startswith("#"):
                categoria, texto = linea.split("|")
                ENTRADAS.append((categoria, texto))

print("Entradas leídas:", len(ENTRADAS))

El `EntityRuler` pide las reglas con una forma precisa: un diccionario por regla, con
`label` —la categoría— y `pattern` —el texto exacto a buscar.

In [ ]:
patrones = []

for categoria, texto in ENTRADAS:
    patron = {"label": categoria, "pattern": texto}
    patrones.append(patron)

reglas.add_patterns(patrones)

cantidad_de_patrones = len(patrones)
print("Patrones cargados:", cantidad_de_patrones)

Y la comparación. **La frase no es inventada**: son dos fragmentos reales de la
transcripción, elegidos justamente porque el modelo de fábrica se equivoca en los dos.

Si eligiéramos una frase donde el modelo ya acierta, las dos salidas darían igual y no se
vería nada. Para auditar una herramienta hay que buscarle los casos donde falla.

No son una muestra aleatoria de todo el audio: están elegidas para hacer visible
un problema que una prueba superficial podría ocultar. Con tu audio, buscá frases
equivalentes en tu propia lista de entidades.

In [ ]:
frase = ("Dice Arturo Jauretzi que esto no era ningún mérito. "
         "Cuando se pronuncia Urquiza Sarmiento se incorpora al ejército.")

doc_original = nlp(frase)
doc_mejorado = nlp_mejorado(frase)

print("--- Modelo de fábrica ---")
for ent in doc_original.ents:
    print("•", ent.text, "->", ent.label_)

print()
print("--- Con nuestras reglas ---")
for ent in doc_mejorado.ents:
    print("•", ent.text, "->", ent.label_)

**Ahí está la diferencia.** El modelo de fábrica se come el verbo dentro del nombre
—`Dice Arturo Jauretzi`— y pega dos personas en una sola entidad —`Urquiza Sarmiento`—.

Nuestras reglas no corrigen la ortografía: **corrigen los límites**. Como el patrón dice
exactamente `Arturo Jauretzi`, el span arranca donde tiene que arrancar. Y como `Urquiza` y
`Sarmiento` son dos patrones distintos, se separan.

Es la misma operación que resuelve las tres formas de *Estados Unidos*: el patrón fija el
recorte, y las tres variantes colapsan en una.

Y lo mismo dibujado, que se lee más rápido que la lista.

In [ ]:
displacy.render(doc_mejorado, style="ent", jupyter=True)

## Volver a contar, ahora con el pipeline con reglas

Una regla que no cambia el resultado no sirve de nada. Rehacemos la auditoría del principio
con `nlp_mejorado`, el pipeline con reglas, y comparamos las dos listas.

El texto que entra es el mismo: `texto_completo`, la transcripción original de Whisper.
Lo que cambió es la herramienta que lo analiza, no el material.

In [ ]:
documento_reglas = nlp_mejorado(texto_completo)

menciones_corregidas = []
for entidad in documento_reglas.ents:
    if entidad.label_ == "PER":
        menciones_corregidas.append(entidad.text)

conteo_con_reglas = Counter(menciones_corregidas)

for nombre, veces in conteo_con_reglas.most_common(15):
    print(veces, "·", nombre)

Compará esa lista con la del principio. Deberían haber desaparecido `Dice Arturo
Jauretzi`, `Gobernaba Rosas` y `Urquiza Sarmiento`.

**Lo que sigue sin resolverse:** `Sarmiento` y `Domingo Faustino Sarmiento` siguen contando
como dos. Son la misma persona y el `EntityRuler` no lo sabe: él marca límites, no unifica
identidades.

### ✎ Para pensar

Ya viste las tres capas funcionando. Ahora la pregunta que importa:

- Si tu objetivo es **publicar la transcripción** para que alguien la lea, ¿en qué capa
  corregís?
- Si tu objetivo es **contar menciones** para un análisis, ¿cambia la respuesta?
- Si tu objetivo es que **otra persona pueda reproducir tu trabajo**, ¿cuál de las tres deja
  el rastro más claro de lo que tocaste?

No hay una respuesta correcta. Hay una decisión, y hay que poder defenderla.

> **Fin del recorrido mínimo.** Si llegaste hasta acá con las tres capas probadas y
> documentadas, ya trabajaste el problema central del cuaderno. Lo que sigue es el
> recorrido extendido.

## Recorrido extendido: normalizar identidades

Volvamos al problema que quedó abierto: `Sarmiento` y `Domingo Faustino Sarmiento`
siguen contando como dos. Para unificarlos hace falta un componente propio, como el
`detector_personajes` del `05`.

Primero, la tabla de equivalencias.

In [ ]:
FORMAS_CANONICAS = {
    "Domingo Faustino Sarmiento": "Sarmiento",
    "Juan Manuel de Rosas": "Rosas",
    "Juan Facundo Quiroga": "Facundo Quiroga",
    "Juan Bautista Alberti": "Alberdi",
    "Juan Bautista Alverde": "Alberdi",
    "Alberti": "Alberdi",
    "el Sonda": "El Zonda",
    "Chivicol": "Chivilcoy",
}

print("Equivalencias declaradas:", len(FORMAS_CANONICAS))

Esta tabla no “descubre” automáticamente quién es quién. Declara una decisión
de análisis: qué formas distintas vamos a contar como la misma entidad.

Incluye personas, pero también puede incluir lugares u organizaciones. Eso importa
porque una red no debería tener un nodo para `Chivicol` y otro para `Chivilcoy`
si decidimos que ambas formas refieren al mismo lugar.

La transcripción original no se modifica. Esta tabla solo indica cómo se
unificarán formas al contar, comparar o construir nodos.

La celda siguiente define `nombre_canonico()`, una función breve que consulta esta
tabla. La vamos a usar dos veces: en el componente de spaCy y al armar la red.

In [ ]:
def nombre_canonico(texto_entidad):
    """Devuelve una forma unificada si la entidad está en el diccionario."""

    if texto_entidad in FORMAS_CANONICAS:
        return FORMAS_CANONICAS[texto_entidad]

    return texto_entidad


print("Prueba:", nombre_canonico("Juan Bautista Alverde"), "·", nombre_canonico("Mitre"))

`nombre_canonico(texto_entidad)` consulta el diccionario de formas canónicas.

- Si la forma está declarada, devuelve la versión unificada: `Juan Bautista Alverde`
  pasa a `Alberdi`.
- Si no está declarada, devuelve el mismo texto: `Mitre` sigue siendo `Mitre`.

Así no necesitamos listar todas las entidades posibles: solo las equivalencias
que decidimos intervenir. Esta decisión debe quedar documentada, porque contar
dos menciones como una sola entidad es una operación de análisis, no un hecho
automático.

El componente siguiente lee las entidades que dejaron `EntityRuler` y NER,
busca cada forma en `FORMAS_CANONICAS` y guarda una versión unificada de las
personas en un atributo propio del documento.

No busca entidades nuevas ni modifica el texto. Su tarea es preparar una lista
de menciones para contar: por ejemplo, decidir que `Sarmiento` y
`Domingo Faustino Sarmiento` se contabilicen bajo una misma forma.

Va al final del pipeline porque necesita que los componentes anteriores ya hayan
reconocido las entidades.

In [ ]:
from spacy.language import Language
from spacy.tokens import Doc

Doc.set_extension("personas", default=None, force=True)


@Language.component("normalizador_de_nombres")
def normalizador_de_nombres(doc):
    """Guarda en doc._.personas los nombres de persona ya unificados."""

    personas_encontradas = []

    for entidad in doc.ents:
        if entidad.label_ == "PER":
            forma_unificada = nombre_canonico(entidad.text)
            personas_encontradas.append(forma_unificada)

    doc._.personas = personas_encontradas

    return doc

El componente quedó definido, pero todavía no forma parte de ningún pipeline.

Ahora lo enchufamos. `last=True` lo pone al final de todo, porque necesita que
el `entity_ruler` y el `ner` ya hayan hecho su trabajo.

In [ ]:
nlp_mejorado.add_pipe("normalizador_de_nombres", last=True)

print("Pipeline:", nlp_mejorado.pipe_names)

> **Si volvés a correr esa celda, va a dar error.** Un pipeline no admite dos
> componentes con el mismo nombre, y spaCy corta con `already exists`. No está roto: te está
> avisando que el componente ya estaba puesto.
>
> Si te pasa, andá a *Entorno de ejecución → Reiniciar y ejecutar todo*. Es la misma
> disciplina de siempre: un cuaderno tiene que correr entero de arriba a abajo.

Y ahora sí, la cuenta final, leyendo el atributo propio en vez de las entidades
crudas.

In [ ]:
documento_final = nlp_mejorado(texto_completo)

conteo_final = Counter(documento_final._.personas)

for nombre, veces in conteo_final.most_common(12):
    print(veces, "·", nombre)

### ✎ Para pensar

- ¿Cuántas menciones tiene Sarmiento ahora contra las que tenía al principio?
- **`FORMAS_CANONICAS` es una decisión tuya, no un dato.** Decidiste que *Alberti* y *Alverde* son
  la misma persona. ¿Con qué te respaldás? ¿Y si dos personas del relato compartieran
  apellido?
- Esa decisión va escrita en la bitácora, igual que la lista de `nombres.txt`.

## Mirar la lengua, no solo los nombres

displaCy también dibuja la sintaxis. Pero acá hay un problema propio de este material, y es
contenido del cuaderno:

**la puntuación de una transcripción la inventó Whisper.** Nadie dicta comas. Dónde corta
una oración es una decisión del modelo de audio, y spaCy va a analizar esas oraciones como
si fueran las que alguien escribió.

Por eso elegimos una frase corta a mano. Con una oración larga el árbol no se lee.

In [ ]:
frase_corta = "Sarmiento fundó un periódico en San Juan."

doc_sintaxis = nlp(frase_corta)

displacy.render(doc_sintaxis, style="dep", jupyter=True,
                options={"compact": True, "distance": 90})

Las flechas van del núcleo a lo que depende de él. `fundó` es la raíz; *Sarmiento* es
su sujeto y *periódico* su objeto directo. `compact=True` achica el dibujo y `distance`
controla cuánto se separan las palabras: sin eso, una oración de quince palabras se va de
pantalla.

### ✎ Para pensar

Buscá en tu transcripción la oración más larga que encuentres y pasala por esta misma celda.

- ¿Se puede leer el árbol?
- ¿Esa oración es realmente una oración, o son tres que el modelo pegó?

## Tres versiones del mismo material

Hasta acá trabajamos con capas distintas. Conviene separarlas antes de construir
la red, porque no todas modifican lo mismo.

| Versión | Texto que se analiza | Pipeline | Qué se intervino |
|---|---|---|---|
| **A** | `texto_completo` | `nlp` | Nada: transcripción original de Whisper y NER de fábrica |
| **B** | `texto_completo` | `nlp_mejorado` | Reglas de entidad y formas canónicas declaradas para analizar |
| **C** | `texto_corregido` | `nlp_mejorado` | Texto intervenido por diccionario, más reglas y formas canónicas |

La primera red que vamos a construir corresponde a la **versión B**.

Eso significa que trabajamos con la transcripción original de Whisper: no la
reescribimos. Pero tampoco usamos spaCy “de fábrica”: aplicamos reglas para
delimitar entidades y unificamos algunas formas para contar y graficar.

Esta distinción importa:

- El `EntityRuler` y las formas canónicas modifican cómo analizamos el texto.
- El diccionario de reemplazos modifica directamente el texto.
- Ninguna de las dos decisiones es neutral ni necesariamente superior: sirven
  para propósitos distintos y deben quedar documentadas.

## La red: quién aparece con quién

Última parte, y la más vistosa. Si dos nombres aparecen seguido en el mismo segmento, algo
los relaciona: el relato los pone juntos.

Es el mismo procedimiento del `09` de la unidad 4, sobre tu propio audio. Sin librería de
grafos: `numpy` para la geometría y `matplotlib` para dibujar.

## La ventana de coocurrencia

Coocurrir significa aparecer juntos dentro de una ventana. Si tomáramos todo
el texto como una sola ventana, todas las entidades coocurrirían con todas y la
red no diría casi nada.

En este cuaderno usamos como ventana cada segmento de Whisper: un tramo breve
de habla con un inicio y un final temporal.

Es una decisión razonable, pero no neutral:

- En la unidad anterior la ventana podía ser el párrafo.
- Acá la ventana es un segmento generado por el modelo de audio.
- Si Whisper corta los segmentos en otros lugares, cambian los pares y cambia
  la red.

La red no representa una relación “natural” entre entidades. Representa qué
entidades quedaron juntas según una forma particular de segmentar y analizar
la transcripción.

> **Ojo con esto:** la ventana la eligió el modelo de audio, no vos.

## Qué versión usa esta red

Esta primera red usa la **versión B**:

```text
texto original de Whisper
    +
pipeline con EntityRuler
    +
formas canónicas declaradas
```

La transcripción original de Whisper no fue reescrita. Sin embargo, no usamos el
pipeline de fábrica: usamos `nlp_mejorado` y formas canónicas declaradas para analizar
entidades. Esto corrige el análisis, no el texto original.

No usamos el pipeline de fábrica porque queremos evitar nodos que sean errores
de límite como `Dice Arturo Jauretzi` o `Urquiza Sarmiento`. Tampoco usamos todavía
`texto_corregido`.

`nlp.pipe()` procesa varios segmentos por lote. Es más rápido que llamar a
`nlp(...)` uno por uno, pero produce el mismo tipo de documento spaCy.
`batch_size=50` es cuántos segmentos entran por lote: cambia la velocidad, no el resultado.

In [ ]:
textos_de_segmentos = []

for segmento in segmentos:
    textos_de_segmentos.append(segmento["texto"])

entidades_por_segmento = []

for documento_de_segmento in nlp_mejorado.pipe(
    textos_de_segmentos,
    batch_size=50
):
    entidades_del_segmento = []

    for entidad in documento_de_segmento.ents:
        if entidad.label_ in ("PER", "LOC", "ORG"):
            forma_unificada = nombre_canonico(entidad.text)
            entidades_del_segmento.append(forma_unificada)

    entidades_por_segmento.append(entidades_del_segmento)

print("Segmentos procesados:", len(entidades_por_segmento))

En este punto ocurren dos operaciones distintas:

1. `EntityRuler` ayuda a decidir qué tramo cuenta como entidad y qué etiqueta
   recibe.
2. `nombre_canonico()` decide qué formas diferentes vamos a agrupar bajo un
   mismo nombre al construir nodos.

Por ejemplo, una regla puede reconocer `Juan Bautista Alverde` como persona,
mientras que la forma canónica puede decidir contar esa mención bajo `Alberdi`.

La primera operación trabaja sobre límites y categorías. La segunda trabaja
sobre equivalencias para el análisis. Ninguna reescribe el texto original de
Whisper.

Ahora contamos pares. Primero quitamos repeticiones dentro de un mismo segmento:
si `Sarmiento` aparece dos veces en un tramo, no queremos contar dos veces el
mismo par con otra entidad dentro de esa misma ventana.

Después ordenamos alfabéticamente las entidades. La comparación
`entidad_a < entidad_b` no mide importancia: solo evita contar dos veces el
mismo vínculo, una vez como `A–B` y otra como `B–A`.

In [ ]:
coocurrencias = Counter()

for lista_de_entidades in entidades_por_segmento:
    entidades_unicas = sorted(set(lista_de_entidades))
    for entidad_a in entidades_unicas:
        for entidad_b in entidades_unicas:
            si_es_un_par_nuevo = entidad_a < entidad_b
            if si_es_un_par_nuevo:
                coocurrencias[(entidad_a, entidad_b)] += 1

print("Pares distintos:", len(coocurrencias))

`most_common` devuelve los pares ordenados de mayor a menor. Los pasamos a una tabla
de pandas para poder mirarlos cómodos.

In [ ]:
quince_pares = coocurrencias.most_common(15)

filas_de_pares = []
for par, veces_juntos in quince_pares:
    entidad_a, entidad_b = par
    filas_de_pares.append({"entidad A": entidad_a, "entidad B": entidad_b,
                           "veces juntas": veces_juntos})

pd.DataFrame(filas_de_pares)

Para dibujar hace falta elegir a quiénes. Tomamos las catorce entidades más
mencionadas: con más, el gráfico se vuelve una maraña.

**Y ahí hay una decisión escondida.** Recortar a catorce no es un ajuste estético: es
afirmar que el resto no importa para esta pregunta. Alguien que mire el gráfico va a ver
catorce nombres y va a suponer que son *los* nombres del relato. Si un personaje quedó
afuera por un par de menciones, desapareció de la historia que cuenta tu imagen.

Es lo mismo que viste en la unidad 3 con los gráficos: **una imagen convence más rápido que
una tabla, y por eso hay que auditarla.**

In [ ]:
todas_las_menciones = []

for lista_de_entidades in entidades_por_segmento:
    for nombre in lista_de_entidades:
        todas_las_menciones.append(nombre)

frecuencia_individual = Counter(todas_las_menciones)
catorce_mas_frecuentes = frecuencia_individual.most_common(14)

for nombre, veces in catorce_mas_frecuentes:
    print(veces, "·", nombre)

Cada nodo va en un punto de un círculo. El ángulo se reparte en partes iguales: si
hay catorce nodos, cada uno ocupa un catorceavo de la vuelta.

In [ ]:
nodos = []
for entidad, veces in catorce_mas_frecuentes:
    nodos.append(entidad)

posiciones_de_nodos = {}
cantidad_de_nodos = len(nodos)
posicion_actual = 0

for nodo in nodos:
    angulo = 2 * np.pi * posicion_actual / cantidad_de_nodos
    posiciones_de_nodos[nodo] = (np.cos(angulo), np.sin(angulo))
    posicion_actual = posicion_actual + 1

print("Nodos ubicados:", len(posiciones_de_nodos))

Y el gráfico. Una línea por par, más gruesa cuantas más veces coocurrieron; un punto
por entidad, más grande cuantas más veces se la nombra.

**Todo el dibujo va en una sola llamada, y eso no es capricho.** Un gráfico de matplotlib se
arma sobre una figura abierta, y el cuaderno la cierra y la muestra cuando termina la celda.
Si partiéramos el dibujo en dos celdas, la segunda estaría dibujando sobre una figura que ya
se mostró y se cerró: no se vería nada.

In [ ]:
def dibujar_red(coocurrencias, posiciones, frecuencias):
    """Dibuja la red completa: primero las líneas, después los puntos encima."""
    figura, eje = plt.subplots(figsize=(9, 9))

    for par, veces_juntos in coocurrencias.items():
        entidad_a, entidad_b = par
        if entidad_a in posiciones and entidad_b in posiciones:
            x_a, y_a = posiciones[entidad_a]
            x_b, y_b = posiciones[entidad_b]
            eje.plot([x_a, x_b], [y_a, y_b], color="gray",
                     linewidth=veces_juntos * 0.6, alpha=0.5)

    return figura, eje

Los puntos van en una segunda función, pero **las dos se llaman desde la misma
celda**. `zorder=3` los pone por delante de las líneas.

In [ ]:
def poner_nodos(eje, posiciones, frecuencias):
    """Dibuja un punto por entidad, con su nombre al lado."""
    for nodo, coordenadas in posiciones.items():
        x, y = coordenadas
        tamanio = frecuencias[nodo] * 20
        eje.scatter(x, y, s=tamanio, color="steelblue",
                    zorder=3, edgecolors="white")
        eje.annotate(nodo, (x, y), fontsize=9, ha="center", va="bottom")

    eje.set_title("Quién aparece con quién por segmento (versión B)")
    eje.axis("off")

Y acá se arma todo junto: figura, líneas, puntos y `show()`, en una sola celda.

In [ ]:
figura, eje = dibujar_red(coocurrencias, posiciones_de_nodos, frecuencia_individual)
poner_nodos(eje, posiciones_de_nodos, frecuencia_individual)

plt.tight_layout()
plt.show()

### ✎ Para pensar

- ¿Las líneas más gruesas unen entidades que efectivamente tienen relación en
  el relato?
- ¿Hay algún nodo que sigue siendo dudoso porque proviene de una transcripción
  o una normalización discutible?
- ¿Qué cambia si modificás la ventana y usás dos segmentos consecutivos en vez
  de uno?

La red que acabás de ver es la **versión B**:

```text
texto_completo + nlp_mejorado + formas canónicas declaradas
```

Como extensión, repetí el procedimiento usando `texto_corregido` en lugar de
`texto_completo`. Esa sería la **versión C**:

```text
texto_corregido + nlp_mejorado + formas canónicas declaradas
```

Una pista: la red trabaja por segmento y `texto_corregido` es un texto corrido. Para
armar la versión C, aplicá los reemplazos de `CORRECCIONES` a cada texto de
`textos_de_segmentos` antes de pasarlo por `nlp_mejorado.pipe()`.

Compará nodos, pares y tamaños. Después respondé:

- ¿Qué cambia cuando intervenís directamente el texto antes de analizarlo?
- ¿Qué correcciones quedan visibles en el texto y cuáles quedan solo en las
  reglas o en el diccionario de formas canónicas?
- ¿Cuál de las dos versiones deja un rastro más claro de lo que modificaste?
- ¿Cuál altera más el material de partida?

No hay una versión automáticamente superior. Depende del propósito: publicar
una transcripción legible, contar menciones, construir una red o dejar un
registro reproducible de las decisiones tomadas.

## Cierre

### Lo que hiciste

- Pasaste una transcripción por spaCy y auditaste su salida.
- Reconstruiste la procedencia de errores de entidad: algunos **heredados del ASR**,
  otros nacidos de la **interacción** entre una transcripción oral y un NER entrenado
  con prosa escrita.
- Probaste tres formas de corregir el mismo error, en tres capas distintas, y viste qué
  arriesga cada una.
- Armaste una red de coocurrencia sobre tu propio material, sabiendo qué versión del
  material y del análisis representa.

### Lo que este texto ya no puede contestar

Esto es lo más importante y lo más fácil de olvidar. La transcripción conserva las palabras
y **pierde todo lo demás**:

- **Quién habló.** Si hay dos voces, el texto no las distingue.
- **Cómo lo dijo.** La ironía, el énfasis, la duda, la broma. Nada de eso sobrevive.
- **Dónde se trabó.** Las pausas y las correcciones sobre la marcha se limpiaron.

Un análisis de frecuencias sobre este texto te va a dar números correctos sobre un objeto
que ya no es el evento oral. **Eso no lo arregla ningún modelo mejor**: se perdió en la
conversión, y la única forma de recuperarlo es volver al audio.

Por eso el `01` guardaba el `segmentos.json` con los tiempos. Para poder volver.

Las capas de corrección que probaste no recuperan automáticamente lo perdido.
Un diccionario puede cambiar una grafía, una regla puede ajustar una entidad y
una forma canónica puede unificar conteos. Ninguna de esas operaciones devuelve
la prosodia, la atribución de voz, los solapamientos ni el contexto del evento oral.

Por eso corregir no significa simplemente “mejorar”. Significa decidir qué
objeto se necesita para una tarea y dejar registrado qué transformaciones se
hicieron para construirlo.